# 06 - Arm 2: DAP + Few-Shot Fine-Tuning



Results append to `results/fewshot_finetune_dap_results.csv`.

In [ ]:
%pip install -q transformers datasets accelerate seqeval

In [ ]:
import gc
import json
import sys
import time
from pathlib import Path

import pandas as pd
if Path('/content').exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

for cand in [Path('/content/drive/MyDrive/AAI590/utils'), Path.cwd(),
             Path.cwd() / 'utils', Path.cwd().parent / 'utils']:
    if (cand / 'ner_common_utils.py').exists():
        sys.path.append(str(cand))
        break
import ner_common_utils as ncu

processed_dir = ncu.resolve_processed_dir()
OUTPUT_ROOT = ncu.resolve_output_root(processed_dir)
fewshot_dir = processed_dir / 'fewshot_splits'
print('processed dir:', processed_dir)
print('output root  :', OUTPUT_ROOT)

In [ ]:
SMOKE_TEST = False

LEARNING_RATE = 5e-5
BATCH_SIZE = 8
EPOCHS = 20          # small train sets need many passes; fixed for every budget
MAX_LENGTH = 256
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# no early stopping and no dev-set model selection, on purpose: under a k-label

METHOD = 'dap_fewshot_finetune'
suffix = '_smoke' if SMOKE_TEST else ''
results_dir = OUTPUT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = results_dir / f'fewshot_finetune_dap_results{suffix}.csv'
print('SMOKE_TEST:', SMOKE_TEST, '| results ->', RESULTS_CSV)

In [ ]:
def checkpoint_for(ds_name):
    d = OUTPUT_ROOT / 'models' / f'dap_{ds_name}'
    if not d.exists():
        d = OUTPUT_ROOT / 'models' / f'dap_{ds_name}_smoke'
        print(f'WARNING: real dap_{ds_name} not found, using the smoke checkpoint')
    assert d.exists(), 'run notebook 04 first'
    return d


for _ds in ncu.TARGET_DATASETS:
    print(_ds, '->', checkpoint_for(_ds))

In [ ]:
# native label set per target dataset, taken from the FULL train file
label_sets, eval_subsets, full_tests = {}, {}, {}
for ds_name in ncu.TARGET_DATASETS:
    train_full = ncu.load_jsonl(processed_dir / ds_name / f'{ds_name}_train.jsonl')
    label_sets[ds_name] = sorted({t for r in train_full for t in r['tags']})
    eval_subsets[ds_name] = ncu.make_or_load_eval_subset(processed_dir, ds_name)
    full_tests[ds_name] = ncu.load_jsonl(processed_dir / ds_name / f'{ds_name}_test.jsonl')
    print(f"{ds_name}: {len(label_sets[ds_name])} labels, "
          f"{len(eval_subsets[ds_name])} eval-subset sentences, "
          f"{len(full_tests[ds_name])} full-test sentences")

In [ ]:
import torch
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments)

grid = [(d, b, s) for d in ncu.TARGET_DATASETS for b in ncu.BUDGETS for s in ncu.SEEDS]
if SMOKE_TEST:
    grid = [('wnut17', 50, 42)]
done = set()
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev['dataset'], prev['budget'], prev['seed']))
    print(f'found {len(done)} finished runs in {RESULTS_CSV.name}, skipping those')

for ds_name, budget, seed in grid:
    if (ds_name, budget, seed) in done:
        continue
    run_name = f'{ds_name} budget={budget} seed={seed}'
    ckpt = checkpoint_for(ds_name)

    split_fp = fewshot_dir / ds_name / f'{ds_name}_train_{budget}_seed_{seed}.jsonl'
    train_rows = ncu.load_jsonl(split_fp)

    labels = label_sets[ds_name]
    label2id, id2label = ncu.build_label_maps(labels)

    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    train_ds = ncu.tokenize_and_align(train_rows, tokenizer, label2id, max_length=MAX_LENGTH)

    ncu.set_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained(
        ckpt, num_labels=len(labels), id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)

    args = TrainingArguments(
        output_dir=str(OUTPUT_ROOT / 'tmp_trainer' / f'{METHOD}_{ds_name}_{budget}_{seed}'),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS if not SMOKE_TEST else 2,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        logging_strategy='steps',
        logging_steps=100,
        save_strategy='no',
        report_to='none',
        disable_tqdm=True,
        seed=seed,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
                      processing_class=tokenizer)

    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0

    sub_rows = eval_subsets[ds_name] if not SMOKE_TEST else eval_subsets[ds_name][:40]
    sub_preds = ncu.predict_tags(model, tokenizer, sub_rows, max_length=MAX_LENGTH)
    m_sub = ncu.compute_entity_f1([r['tags'] for r in sub_rows], sub_preds)

    m_full = None
    if not SMOKE_TEST:
        full_rows = full_tests[ds_name]
        full_preds = ncu.predict_tags(model, tokenizer, full_rows, max_length=MAX_LENGTH)
        m_full = ncu.compute_entity_f1([r['tags'] for r in full_rows], full_preds)

    row = {
        'method': METHOD, 'dataset': ds_name, 'budget': budget, 'seed': seed,
        'precision': m_sub['precision'], 'recall': m_sub['recall'], 'f1': m_sub['f1'],
        'support': m_sub['support'], 'n_eval': len(sub_rows),
        'f1_full_test': m_full['f1'] if m_full else None,
        'precision_full_test': m_full['precision'] if m_full else None,
        'recall_full_test': m_full['recall'] if m_full else None,
        'train_seconds': round(train_seconds, 1),
        'epochs': args.num_train_epochs, 'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE, 'n_train': len(train_rows),
        'base_checkpoint': str(ckpt), 'device': ncu.pick_device(),
        'per_type': json.dumps(m_sub['per_type']),
    }
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', index=False,
                               header=not RESULTS_CSV.exists())
    print(f"{run_name}: F1={m_sub['f1']:.3f} (subset) "
          f"{('F1=%.3f (full test) ' % m_full['f1']) if m_full else ''}"
          f"in {train_seconds:.0f}s train")

    # keep memory in check between runs
    del trainer, model
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\nall runs finished')

In [ ]:
results = pd.read_csv(RESULTS_CSV)
cols = ['dataset', 'budget', 'seed', 'precision', 'recall', 'f1', 'f1_full_test',
        'train_seconds']
results[cols].sort_values(['dataset', 'budget', 'seed']).round(3)